# Neyman Orthogonal Scores: Why DML Tolerates Imperfect ML

**Goal:** See that DML's orthogonal score makes it robust to first-stage ML errors, while naive plug-in diverges.

**Time:** 15 minutes

You will intentionally degrade ML predictions and watch DML stay approximately unbiased
while the naive plug-in estimator falls apart.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## Setup: Known DGP

Simple linear DGP so we can clearly see the orthogonality effect without confounding from model misspecification.

In [ ]:
n = 5000
p = 20
true_theta = 1.0

X = np.random.randn(n, p)
D = 0.5 * X[:, 0] + 0.3 * X[:, 1] + np.random.randn(n) * 0.5
Y = true_theta * D + X[:, 0] + 0.5 * X[:, 2] + np.random.randn(n)

print(f'n={n}, p={p}, true_theta={true_theta}')

## DML vs Naive Plug-in with Degraded ML

We add increasing levels of Gaussian noise to the ML predictions to simulate worse first-stage models.
DML residualises both Y and D; the naive approach only residualises Y.

In [ ]:
def dml_estimate(Y, D, X, noise_level=0.0, seed=42):
    """DML with optional noise injection."""
    n = len(Y)
    rY, rD = np.zeros(n), np.zeros(n)
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    rng = np.random.RandomState(seed + 1000)

    for train_idx, test_idx in kf.split(X):
        rf_y = RandomForestRegressor(100, random_state=seed)
        rf_d = RandomForestRegressor(100, random_state=seed)
        rf_y.fit(X[train_idx], Y[train_idx])
        rf_d.fit(X[train_idx], D[train_idx])

        rY[test_idx] = Y[test_idx] - rf_y.predict(X[test_idx]) - noise_level * rng.randn(len(test_idx))
        rD[test_idx] = D[test_idx] - rf_d.predict(X[test_idx]) - noise_level * rng.randn(len(test_idx))

    return np.sum(rD * rY) / np.sum(rD ** 2)

def naive_estimate(Y, D, X, noise_level=0.0, seed=42):
    """Naive: only residualise Y, use raw D."""
    n = len(Y)
    rY = np.zeros(n)
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    rng = np.random.RandomState(seed + 2000)

    for train_idx, test_idx in kf.split(X):
        rf_y = RandomForestRegressor(100, random_state=seed)
        rf_y.fit(X[train_idx], Y[train_idx])
        rY[test_idx] = Y[test_idx] - rf_y.predict(X[test_idx]) - noise_level * rng.randn(len(test_idx))

    return np.sum(D * rY) / np.sum(D ** 2)

noise_levels = [0.0, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
dml_results = [dml_estimate(Y, D, X, nl) for nl in noise_levels]
naive_results = [naive_estimate(Y, D, X, nl) for nl in noise_levels]

print(f'True effect: {true_theta:.2f}\n')
print(f'{"Noise":<8} {"DML":>8} {"Naive":>8} {"DML Bias":>10} {"Naive Bias":>12}')
print('=' * 52)
for nl, d, na in zip(noise_levels, dml_results, naive_results):
    print(f'{nl:<8.2f} {d:>8.3f} {na:>8.3f} {d-true_theta:>+10.3f} {na-true_theta:>+12.3f}')

## Visualise: DML vs Naive Sensitivity

Plot the bias of each method as a function of first-stage noise level.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

dml_biases = [d - true_theta for d in dml_results]
naive_biases = [na - true_theta for na in naive_results]

ax.plot(noise_levels, np.abs(dml_biases), 'o-', linewidth=2, markersize=8,
        color='forestgreen', label='DML (orthogonal score)')
ax.plot(noise_levels, np.abs(naive_biases), 's-', linewidth=2, markersize=8,
        color='crimson', label='Naive plug-in')
ax.set_xlabel('First-Stage Noise Level', fontsize=12)
ax.set_ylabel('|Bias|', fontsize=12)
ax.set_title('Orthogonal Score: Robust to First-Stage Errors', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
ax.set_yscale('log')

plt.tight_layout()
plt.show()

print('DML bias grows much slower than naive bias.')
print('This is the orthogonality property: Bias proportional to error^2, not error.')

## Monte Carlo: Robustness Across Simulations

Run 50 simulations at each noise level to get reliable bias estimates with confidence bands.

In [ ]:
n_sims = 50
test_noise = [0.0, 0.1, 0.3, 0.5, 1.0]
mc_dml = {nl: [] for nl in test_noise}
mc_naive = {nl: [] for nl in test_noise}

for sim in range(n_sims):
    rng_s = np.random.RandomState(sim)
    X_s = rng_s.randn(n, p)
    D_s = 0.5 * X_s[:, 0] + 0.3 * X_s[:, 1] + rng_s.randn(n) * 0.5
    Y_s = true_theta * D_s + X_s[:, 0] + 0.5 * X_s[:, 2] + rng_s.randn(n)

    for nl in test_noise:
        mc_dml[nl].append(dml_estimate(Y_s, D_s, X_s, nl, seed=sim))
        mc_naive[nl].append(naive_estimate(Y_s, D_s, X_s, nl, seed=sim))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dml_means = [np.mean(mc_dml[nl]) - true_theta for nl in test_noise]
dml_sds = [np.std(mc_dml[nl]) for nl in test_noise]
naive_means = [np.mean(mc_naive[nl]) - true_theta for nl in test_noise]
naive_sds = [np.std(mc_naive[nl]) for nl in test_noise]

axes[0].errorbar(test_noise, dml_means, yerr=dml_sds, fmt='o-',
                 color='forestgreen', capsize=5, linewidth=2, label='DML')
axes[0].axhline(y=0, color='black', linestyle='--')
axes[0].set_xlabel('Noise Level', fontsize=12)
axes[0].set_ylabel('Bias', fontsize=12)
axes[0].set_title('DML Bias (50 simulations)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

axes[1].errorbar(test_noise, naive_means, yerr=naive_sds, fmt='s-',
                 color='crimson', capsize=5, linewidth=2, label='Naive')
axes[1].axhline(y=0, color='black', linestyle='--')
axes[1].set_xlabel('Noise Level', fontsize=12)
axes[1].set_ylabel('Bias', fontsize=12)
axes[1].set_title('Naive Bias (50 simulations)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

**What you learned:**
1. Naive plug-in estimators have first-order sensitivity to ML errors (bias proportional to error)
2. DML's orthogonal score gives second-order sensitivity (bias proportional to error squared)
3. The "double" in DML means residualising BOTH Y and D -- this is what creates orthogonality
4. DML remains approximately unbiased even with substantial first-stage noise

**What is next:**
- **Module 04:** Cross-fitting -- the second essential ingredient for valid DML inference
- **Module 05:** Putting it all together with `doubleml`

**Key takeaway:** You do not need perfect ML models for valid causal inference.
The orthogonal score ensures that decent ML is enough.